In [96]:
import pandas as pd

In [97]:
df = pd.read_csv('expenses.csv', thousands=',')
people = ['ben','sherina','keiton','chris','hyeok','mandy','eric', 'harry','haeyung']

df = df.melt(id_vars=[col for col in df.columns if col not in people], var_name='owed_by', value_name='portion')

# Check data types of relevant columns
print(df[['amount', 'portion', 'TOTAL']].dtypes)
df['amount'] = df['amount'].astype(float)

# Ensure numeric types for calculations
df['amount'] = pd.to_numeric(df['amount'], errors='ignore')
df['portion'] = pd.to_numeric(df['portion'], errors='ignore')
df['TOTAL'] = pd.to_numeric(df['TOTAL'], errors='ignore')

#lower case and trim whitespace
df['paid_by'] = df['paid_by'].str.lower().str.strip()
df['owed_by'] = df['owed_by'].str.lower().str.strip()


df = df[(df['portion']!=0) & (df['portion'].notna())]

amount     float64
portion    float64
TOTAL      float64
dtype: object


In [98]:
df['owed_amount'] = df['amount'] * df['portion'] / df['TOTAL']

In [99]:
for person in people:
    print(f"{person} owes: {df[df['owed_by']==person]['owed_amount'].sum():.2f}")


ben owes: 1142.39
sherina owes: 1141.41
keiton owes: 1123.73
chris owes: 1183.22
hyeok owes: 1201.15
mandy owes: 280.02
eric owes: 315.45
harry owes: 313.36
haeyung owes: 344.42


In [100]:
for person in people:
    print(f"{person} paid: {df[df['paid_by']==person]['owed_amount'].sum():.2f}")


ben paid: 5250.78
sherina paid: 328.29
keiton paid: 582.54
chris paid: 554.57
hyeok paid: 49.33
mandy paid: 0.00
eric paid: 279.65
harry paid: 0.00
haeyung paid: 0.00


In [101]:



df1 = df[df['owed_amount']!=0]
df1 = df1[df1['paid_by']!=df1['owed_by']]

mapping = {'sherina': 'ben', 'haeyung': 'keiton', 'mandy':'eric'}
df1['owed_by'] = df1['owed_by'].replace(mapping)
df1['paid_by'] = df1['paid_by'].replace(mapping)

df1.reset_index(drop = True, inplace=True)
print(len(df1), 'transactions to do')
df1.head(3)

130 transactions to do


,Item,description,amount,paid_by,TOTAL,owed_by,portion,owed_amount
0,Suburu + fuel,by /person/day,582.54,keiton,1.0,ben,0.160377,93.426226
1,Ramen,NaN,136.00,chris,5.0,ben,1.000000,27.200000
2,Trader Joes,NaN,49.33,hyeok,5.0,ben,1.000000,9.866000


In [102]:
from collections import defaultdict
# Step 1: Calculate net balances for each person
balances = defaultdict(float)
for _, row in df1.iterrows():
    balances[row['paid_by']] += row['owed_amount']
    balances[row['owed_by']] -= row['owed_amount']
balances

defaultdict(float,
            {'keiton': -885.6084446945404,
             'ben': nan,
             'chris': -628.6523022644456,
             'hyeok': nan,
             'eric': -315.8219017760678,
             'harry': -313.36484536998466})

In [103]:
 # Step 2: Separate into creditors and debtors
creditors = []
debtors = []
for person, balance in balances.items():
    if balance > 0:
        creditors.append((person, balance))
    elif balance < 0:
        debtors.append((person, -balance))
creditors.sort(key=lambda x: x[1], reverse=True)
debtors.sort(key=lambda x: x[1], reverse=True)

In [104]:
# Step 3: Minimize transactions
minimized_transactions = []
while creditors and debtors:
    creditor, credit_amount = creditors.pop()
    debtor, debt_amount = debtors.pop()

    payment = min(credit_amount, debt_amount)
    minimized_transactions.append({'paid_by': creditor, 'owed_by': debtor, 'owed_amount': payment})

    if credit_amount > payment:
        creditors.append((creditor, credit_amount - payment))
        creditors.sort(key=lambda x: x[1], reverse=True)
    if debt_amount > payment:
        debtors.append((debtor, debt_amount - payment))
        debtors.sort(key=lambda x: x[1], reverse=True)


In [105]:
for transaction in minimized_transactions:
    print(transaction['owed_by'], 'pays', round(transaction['owed_amount'],2), 'to', transaction['paid_by'])

In [106]:
df

,Item,description,amount,paid_by,TOTAL,owed_by,portion,owed_amount
0,BNB,by /person/nights,2758.52,ben,1.0,ben,0.164948,455.013609
1,Suburu + fuel,by /person/day,582.54,keiton,1.0,ben,0.160377,93.426226
2,Van + fuel,by /person/day,1383.20,ben,1.0,ben,0.160377,221.833962
3,Ramen,NaN,136.00,chris,5.0,ben,1.000000,27.200000
4,Trader Joes,NaN,49.33,hyeok,5.0,ben,1.000000,9.866000
...,...,...,...,...,...,...,...,...
257,Suburu + fuel,by /person/day,582.54,keiton,1.0,eric,0.047170,27.478302
258,Van + fuel,by /person/day,1383.20,ben,1.0,eric,0.047170,65.245283
277,mensho,NaN,279.65,eric,1.0,eric,0.211538,59.156731
282,hotpot ombu,NaN,318.95,chris,1.0,eric,0.148586,47.391397


In [107]:
# Generate report to README.md
import datetime

report = "# Expense Splitting Report\n\n"
report += "Google Sheets Link\n"
report += "https://docs.google.com/spreadsheets/d/1sgjZCzSm74SpFO3mT2y9Xk_OrHESxQ3xAgAuiUvRKkQ/edit?gid=1818656043#gid=1818656043\n\n"
report += "owed_amount = amount * portion / total\n\n"
report += f"Report generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"


# Total expenses
total_expenses = df['owed_amount'].sum()
report += f"## Total Expenses\n{total_expenses:.2f}\n\n"

# Amount paid and owed
paid_by_person = df.groupby('paid_by')['owed_amount'].sum()
owed_by_person = df.groupby('owed_by')['owed_amount'].sum()

report += "## Summary by Person\n\n"
report += "| Person | Paid | Owes | Net Balance |\n"
report += "|--------|------|------|-------------|\n"
for person in people:
    paid = paid_by_person.get(person, 0)
    owed = owed_by_person.get(person, 0)
    net = paid-owed
    balance_str = f"{net:.2f}" if net != 0 else "0.00"
    report += f"| {person} | {paid:.2f} | {owed:.2f} | {balance_str} |\n"
report += "\n"

report += "## Minimized Transactions\n"

for sugar_baby, sugar_daddy in mapping.items():
    report += f"Sugar daddy {sugar_daddy} pays for sugar baby {sugar_baby}\n"
for transaction in minimized_transactions:
    report += f"- {transaction['owed_by']} pays {round(transaction['owed_amount'],2)} to {transaction['paid_by']}\n"

with open('README.md', 'w') as f:
    f.write(report)
print("Report generated and saved to README.md")

Report generated and saved to README.md
